<a href="https://colab.research.google.com/github/wadhawanakshit/retail-data-pipeline/blob/main/Ecommerce_Sales_Analytics_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import pandas as pd
import numpy as np

# Set seed for reproducibility
np.random.seed(42)
n_rows = 10000

print("Generating raw e-commerce dataset...")
data = {
    'order_id': [f'ORD-{10000 + i}' for i in range(n_rows)],
    'order_date': pd.date_range(start='2025-01-01', end='2025-12-31', periods=n_rows).strftime('%Y-%m-%d'),
    'customer_id': np.random.choice([f'CUST-{i}' for i in range(1, 2501)] + [None], size=n_rows, p=[0.98/2500]*2500 + [0.02]),
    'city': np.random.choice(['Delhi', 'delhi', 'Noida', 'Gurugram', 'Mumbai', 'Bengaluru', 'Bengaluru '], size=n_rows),
    'category': np.random.choice(['Electronics', 'Fashion', 'Home & Kitchen', 'Beauty', 'Books'], size=n_rows, p=[0.25, 0.30, 0.20, 0.15, 0.10]),
    'quantity': np.random.choice([1, 2, 3, 4, -1], size=n_rows, p=[0.55, 0.25, 0.12, 0.06, 0.02]),
    'unit_price': np.round(np.random.uniform(250, 8500, size=n_rows), 2),
    'discount_pct': np.random.choice([0, 5, 10, 15, 25], size=n_rows, p=[0.40, 0.25, 0.20, 0.10, 0.05]),
    'order_status': np.random.choice(['Delivered', 'Returned', 'Cancelled'], size=n_rows, p=[0.82, 0.10, 0.08])
}

df_raw = pd.DataFrame(data)

# Inject deliberate duplicates to simulate logging errors
duplicates = df_raw.sample(150, random_state=42)
df_raw = pd.concat([df_raw, duplicates], ignore_index=True)

# --- DATA QUALITY AUDIT & CLEANING ---
df = df_raw.copy()

print(f"Initial raw rows: {len(df)}")

# 1. Remove duplicate order IDs
df = df.drop_duplicates(subset=['order_id'], keep='first')

# 2. Standardize text formatting (fixes 'delhi' vs 'Delhi' and trailing spaces)
df['city'] = df['city'].str.strip().str.title()

# 3. Handle missing customer IDs (tag as 'GUEST')
df['customer_id'] = df['customer_id'].fillna('GUEST')

# 4. Filter out corrupted negative quantities
df = df[df['quantity'] > 0].copy()

# 5. Parse dates & calculate revenue metrics
df['order_date'] = pd.to_datetime(df['order_date'])
df['order_month'] = df['order_date'].dt.to_period('M').astype(str)
df['gross_revenue'] = np.round(df['quantity'] * df['unit_price'], 2)
df['net_revenue'] = np.round(df['gross_revenue'] * (1 - df['discount_pct'] / 100), 2)

# Save clean file locally in Colab
df.to_csv('clean_ecommerce_orders.csv', index=False)
print(f"Data cleaning complete! Final clean dataset rows: {len(df)}")
display(df.head())

Generating raw e-commerce dataset...
Initial raw rows: 10150
Data cleaning complete! Final clean dataset rows: 9798


,order_id,order_date,customer_id,city,category,quantity,unit_price,discount_pct,order_status,order_month,gross_revenue,net_revenue
0,ORD-10000,2025-01-01,CUST-956,Bengaluru,Fashion,1,3944.40,10,Delivered,2025-01,3944.40,3549.96
1,ORD-10001,2025-01-01,CUST-2426,Gurugram,Fashion,2,4162.09,0,Delivered,2025-01,8324.18,8324.18
2,ORD-10002,2025-01-01,CUST-1868,Delhi,Fashion,1,7638.29,5,Delivered,2025-01,7638.29,7256.38
3,ORD-10003,2025-01-01,CUST-1528,Mumbai,Home & Kitchen,1,1989.91,15,Delivered,2025-01,1989.91,1691.42
4,ORD-10004,2025-01-01,CUST-399,Bengaluru,Home & Kitchen,4,2044.28,0,Cancelled,2025-01,8177.12,8177.12


In [6]:
import sqlite3

# Load clean data and create an in-memory SQLite database
conn = sqlite3.connect('ecommerce_analytics.db')
df.to_sql('orders', conn, if_exists='replace', index=False)

print("Database created successfully and 'orders' table loaded!\n")

# 1. Monthly Revenue & Performance Trend
query_monthly = """
SELECT
    order_month,
    COUNT(order_id) AS total_orders,
    SUM(quantity) AS total_units_sold,
    ROUND(SUM(net_revenue), 2) AS total_net_revenue,
    ROUND(AVG(net_revenue), 2) AS average_order_value
FROM orders
WHERE order_status = 'Delivered'
GROUP BY order_month
ORDER BY order_month;
"""
print("--- 1. Monthly Revenue & Performance Trend ---")
monthly_result = pd.read_sql(query_monthly, conn)
display(monthly_result)

# 2. Top Cities by Revenue
query_cities = """
SELECT
    city,
    COUNT(order_id) AS orders_count,
    ROUND(SUM(net_revenue), 2) AS total_revenue
FROM orders
WHERE order_status = 'Delivered'
GROUP BY city
ORDER BY total_revenue DESC
LIMIT 5;
"""
print("\n--- 2. Top 5 Cities by Revenue ---")
display(pd.read_sql(query_cities, conn))

# 3. Category Performance & Return Rate
query_categories = """
SELECT
    category,
    COUNT(order_id) AS total_orders,
    ROUND(SUM(CASE WHEN order_status = 'Returned' THEN 1.0 ELSE 0.0 END) * 100.0 / COUNT(order_id), 2) AS return_rate_pct,
    ROUND(SUM(net_revenue), 2) AS net_revenue
FROM orders
GROUP BY category
ORDER BY net_revenue DESC;
"""
print("\n--- 3. Category Performance & Return Rate ---")
display(pd.read_sql(query_categories, conn))

conn.close()

Database created successfully and 'orders' table loaded!

--- 1. Monthly Revenue & Performance Trend ---


,order_month,total_orders,total_units_sold,total_net_revenue,average_order_value
0,2025-01,689,1202,4880454.77,7083.39
1,2025-02,608,1021,4089502.97,6726.16
2,2025-03,677,1193,4766949.57,7041.28
3,2025-04,665,1133,4729314.71,7111.75
4,2025-05,701,1171,4884822.12,6968.36
5,2025-06,657,1109,4689657.96,7137.99
6,2025-07,683,1113,4632976.25,6783.27
7,2025-08,679,1126,4553865.63,6706.72
8,2025-09,661,1127,4681916.77,7083.08
9,2025-10,665,1087,4484631.02,6743.81



--- 2. Top 5 Cities by Revenue ---


,city,orders_count,total_revenue
0,Bengaluru,2335,16571573.85
1,Delhi,2291,15603247.49
2,Gurugram,1122,7725225.12
3,Mumbai,1110,7668426.58
4,Noida,1112,7589548.54



--- 3. Category Performance & Return Rate ---


,category,total_orders,return_rate_pct,net_revenue
0,Fashion,2924,10.47,19949085.03
1,Electronics,2463,10.27,17082698.31
2,Home & Kitchen,1978,11.12,13583790.01
3,Beauty,1430,8.18,10431883.49
4,Books,1003,8.77,6905277.88


In [7]:
import sqlite3

conn = sqlite3.connect('ecommerce_analytics.db')

monthly_df = pd.read_sql("SELECT order_month, COUNT(order_id) AS total_orders, ROUND(SUM(net_revenue), 2) AS total_revenue FROM orders WHERE order_status = 'Delivered' GROUP BY order_month", conn)
category_df = pd.read_sql("SELECT category, COUNT(order_id) AS total_orders, ROUND(SUM(net_revenue), 2) AS total_revenue FROM orders GROUP BY category", conn)
city_df = pd.read_sql("SELECT city, COUNT(order_id) AS total_orders, ROUND(SUM(net_revenue), 2) AS total_revenue FROM orders WHERE order_status = 'Delivered' GROUP BY city", conn)

conn.close()

# Export multi-sheet Excel workbook
output_file = 'ecommerce_executive_summary.xlsx'
with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    monthly_df.to_excel(writer, sheet_name='Monthly Performance', index=False)
    category_df.to_excel(writer, sheet_name='Category Summary', index=False)
    city_df.to_excel(writer, sheet_name='City Performance', index=False)

print(f"Success! Report '{output_file}' generated and ready for download.")

Success! Report 'ecommerce_executive_summary.xlsx' generated and ready for download.


In [8]:
import plotly.express as px

# Group data for visualization
chart_df = df[df['order_status'] == 'Delivered'].groupby('order_month')['net_revenue'].sum().reset_index()

# Plot clean bar/line chart
fig = px.bar(chart_df, x='order_month', y='net_revenue',
             title='<b>Monthly Net Revenue Trend (2025)</b>',
             labels={'order_month': 'Month', 'net_revenue': 'Net Revenue (₹)'},
             text_auto=',.2f', color='net_revenue')
fig.update_layout(template='plotly_white', title_x=0.5)
fig.show()